<a href="https://colab.research.google.com/github/Saiji/Data-Science-Work/blob/master/Data_Center_%26_Lab_Operations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import pytest

from capacity_engine import (
    CapacityUtilizationEngine,
    DecommissionTicket,
    Dimension,
    NodeSnapshot,
    PipelineOrder,
    UtilizationZone,
)
from forecasting import (
    ARIMAForecaster,
    CapacityForecastEnsemble,
    ExponentialSmoothingForecaster,
)


# ---------------------------------------------------------------------------
# Fixtures
# ---------------------------------------------------------------------------

@pytest.fixture
def engine():
    return CapacityUtilizationEngine()


@pytest.fixture
def basic_snapshot():
    return NodeSnapshot(
        node_id="TEST-NODE-01",
        dimension=Dimension.SPACE,
        timestamp=datetime.utcnow(),
        allocated_units=380.0,
        installed_capacity=500.0,
    )


@pytest.fixture
def power_snapshot():
    return NodeSnapshot(
        node_id="TEST-NODE-01",
        dimension=Dimension.POWER,
        timestamp=datetime.utcnow(),
        allocated_units=60.0,
        installed_capacity=100.0,
        pue=1.5,
    )


@pytest.fixture
def mock_series():
    """52-week synthetic demand series with upward trend."""
    np.random.seed(0)
    trend = np.linspace(40, 75, 52)
    noise = np.random.normal(0, 2, 52)
    return pd.Series(np.maximum(0, trend + noise))


# ---------------------------------------------------------------------------
# CapacityUtilizationEngine tests
# ---------------------------------------------------------------------------

class TestCapacityUtilizationEngine:

    def test_basic_utilization(self, engine, basic_snapshot):
        """Base formula: (380) / (500 * 0.90) * 100 = 84.4%"""
        result = engine.compute_utilization(basic_snapshot)
        expected = 380 / (500 * 0.90) * 100
        assert abs(result.raw_utilization_pct - expected) < 0.1
        assert result.zone == UtilizationZone.PROCURE

    def test_pipeline_order_adds_to_numerator(self, engine, basic_snapshot):
        order = PipelineOrder(
            order_id="PO-001",
            dimension=Dimension.SPACE,
            capacity_units=50.0,
            expected_live=datetime.utcnow() + timedelta(days=14),
        )
        result_without = engine.compute_utilization(basic_snapshot)
        result_with    = engine.compute_utilization(basic_snapshot, [order])
        assert result_with.committed_pipeline == pytest.approx(50.0, abs=0.1)
        assert result_with.raw_utilization_pct > result_without.raw_utilization_pct

    def test_expired_pipeline_order_ignored(self, engine, basic_snapshot):
        """Orders with expected_live in the past are already counted in allocated — ignore."""
        past_order = PipelineOrder(
            order_id="PO-OLD",
            dimension=Dimension.SPACE,
            capacity_units=50.0,
            expected_live=datetime.utcnow() - timedelta(days=1),  # past
        )
        result = engine.compute_utilization(basic_snapshot, [past_order])
        assert result.committed_pipeline == pytest.approx(0.0, abs=0.01)

    def test_decommission_credit_reduces_allocated(self, engine, basic_snapshot):
        ticket = DecommissionTicket(
            ticket_id="TKT-001",
            dimension=Dimension.SPACE,
            capacity_units=40.0,
            created_at=datetime.utcnow() - timedelta(days=15),
            expected_completion=datetime.utcnow() + timedelta(days=15),
        )
        result = engine.compute_utilization(basic_snapshot, decommission_tickets=[ticket])
        # 15 of 30 days elapsed → 50% credit → 20 RU deducted
        assert result.decommission_credit == pytest.approx(20.0, abs=1.0)
        assert result.allocated < 380.0

    def test_pue_applied_to_power_only(self, engine, power_snapshot, basic_snapshot):
        """PUE correction must apply only to Power dimension."""
        power_result = engine.compute_utilization(power_snapshot)
        space_result = engine.compute_utilization(basic_snapshot)
        # Power: allocated_units * PUE = 60 * 1.5 = 90 (before buffer)
        assert power_result.allocated == pytest.approx(90.0, abs=0.1)
        # Space: no PUE adjustment
        assert space_result.allocated == pytest.approx(380.0, abs=0.1)

    def test_seasonality_weight_clamped(self, engine, basic_snapshot):
        """Seasonality weight outside [0.92, 1.08] must be clamped."""
        result_high = engine.compute_utilization(basic_snapshot, seasonality_weight=2.0)
        result_low  = engine.compute_utilization(basic_snapshot, seasonality_weight=0.1)
        assert result_high.seasonality_weight == pytest.approx(1.08, abs=0.001)
        assert result_low.seasonality_weight  == pytest.approx(0.92, abs=0.001)

    def test_zone_classification(self, engine):
        """Zone boundaries: <70 optimal, 70-80 plan, 80-90 procure, 90+ emergency."""
        def make_snap(allocated, installed=100.0):
            return NodeSnapshot("N", Dimension.NETWORK, datetime.utcnow(),
                                allocated, installed)

        # Reserve buffer for NETWORK is 15% → effective_cap = 85
        # util = allocated / 85 * 100
        assert engine.compute_utilization(make_snap(55)).zone  == UtilizationZone.OPTIMAL
        assert engine.compute_utilization(make_snap(63)).zone  == UtilizationZone.PLAN
        assert engine.compute_utilization(make_snap(72)).zone  == UtilizationZone.PROCURE
        assert engine.compute_utilization(make_snap(85)).zone  == UtilizationZone.EMERGENCY

    def test_zero_division_protection(self, engine):
        """Zero installed_capacity should not raise ZeroDivisionError."""
        snap = NodeSnapshot("N", Dimension.SPACE, datetime.utcnow(), 10.0, 0.0)
        result = engine.compute_utilization(snap)
        assert result.effective_capacity >= 1.0  # clamped to 1.0

    def test_to_dict_all_keys_present(self, engine, basic_snapshot):
        result = engine.compute_utilization(basic_snapshot)
        d = result.to_dict()
        expected_keys = {
            "node_id", "dimension", "timestamp", "allocated",
            "committed_pipeline", "decommission_credit", "installed_capacity",
            "reserve_buffer", "effective_capacity", "raw_utilization_pct",
            "adjusted_utilization_pct", "seasonality_weight", "zone", "formula",
        }
        assert expected_keys.issubset(d.keys())


# ---------------------------------------------------------------------------
# Forecasting tests
# ---------------------------------------------------------------------------

class TestARIMAForecaster:

    def test_output_shape(self, mock_series):
        model = ARIMAForecaster()
        fc    = model.fit_predict(mock_series, horizon=4)
        assert fc.shape == (4,)

    def test_no_negative_values(self, mock_series):
        model = ARIMAForecaster()
        fc    = model.fit_predict(mock_series, horizon=12)
        assert np.all(fc >= 0)


class TestExponentialSmoothingForecaster:

    def test_output_shape(self, mock_series):
        model = ExponentialSmoothingForecaster()
        fc    = model.fit_predict(mock_series, horizon=4)
        assert fc.shape == (4,)

    def test_short_series_handled(self):
        """Should not raise even with only 8 observations."""
        short = pd.Series(np.linspace(10, 20, 8))
        model = ExponentialSmoothingForecaster()
        fc    = model.fit_predict(short, horizon=4)
        assert fc.shape == (4,)


class TestCapacityForecastEnsemble:

    def test_output_shape(self, mock_series):
        ensemble = CapacityForecastEnsemble()
        result   = ensemble.forecast(mock_series, horizon_weeks=12)
        assert result.forecast_values.shape == (12,)
        assert result.lower_bound.shape     == (12,)
        assert result.upper_bound.shape     == (12,)

    def test_lower_le_forecast_le_upper(self, mock_series):
        ensemble = CapacityForecastEnsemble()
        result   = ensemble.forecast(mock_series, horizon_weeks=12)
        assert np.all(result.lower_bound <= result.forecast_values + 1e-6)
        assert np.all(result.forecast_values <= result.upper_bound + 1e-6)

    def test_seasonality_weight_in_bounds(self, mock_series):
        ensemble = CapacityForecastEnsemble()
        result   = ensemble.forecast(mock_series, horizon_weeks=12)
        assert 0.92 <= result.seasonality_weight <= 1.08

    def test_weights_normalised(self):
        ensemble = CapacityForecastEnsemble({"arima": 2, "prophet": 3, "es": 5})
        total = sum(ensemble.weights.values())
        assert abs(total - 1.0) < 1e-9

    def test_insufficient_history_raises(self):
        ensemble = CapacityForecastEnsemble()
        short    = pd.Series([1.0, 2.0, 3.0])
        with pytest.raises(ValueError, match="at least 8"):
            ensemble.forecast(short, horizon_weeks=4)

    def test_mape_returns_float(self, mock_series):
        ensemble = CapacityForecastEnsemble()
        mape     = ensemble.compute_mape(mock_series, test_size=4, horizon_weeks=4)
        assert isinstance(mape, float)
        assert mape >= 0

    def test_to_dataframe(self, mock_series):
        ensemble = CapacityForecastEnsemble()
        result   = ensemble.forecast(mock_series, horizon_weeks=6)
        df       = result.to_dataframe()
        assert len(df) == 6
        assert "forecast" in df.columns
        assert "lower_80" in df.columns


# ---------------------------------------------------------------------------
# Integration test
# ---------------------------------------------------------------------------

class TestEndToEnd:

    def test_pipeline_single_node(self):
        """Smoke test: full utilization computation with mock history."""
        from pipeline import CapacityForecastingPipeline, PipelineConfig

        config   = PipelineConfig(history_source="mock")
        pipeline = CapacityForecastingPipeline(config)

        snapshots = [
            NodeSnapshot("NODE-X", Dimension.SPACE,   datetime.utcnow(), 38, 50),
            NodeSnapshot("NODE-X", Dimension.NETWORK, datetime.utcnow(), 310, 500),
            NodeSnapshot("NODE-X", Dimension.POWER,   datetime.utcnow(), 74, 100, pue=1.45),
            NodeSnapshot("NODE-X", Dimension.COOLING, datetime.utcnow(), 62, 100),
        ]
        history = {
            dim.value: CapacityForecastingPipeline._generate_mock_history(dim)
            for dim in Dimension
        }

        df = pipeline.run_single_node("NODE-X", snapshots, history)

        assert len(df) == 4
        assert set(df["dimension"]) == {d.value for d in Dimension}
        assert df["adjusted_utilization_pct"].between(0, 200).all()
        assert df["zone"].isin([z.value for z in UtilizationZone]).all()

NameError: name '__file__' is not defined